# CoSE 220 · AI & Innovation — the day notebook

One notebook for the whole day. **Part N goes with deck D01.0N**:

| Part | Deck | Time | What you do |
| --- | --- | --- | --- |
| 1 · Get Ready | D01.01 | the day before | check your laptop works |
| 2 · Train Your First Model | D01.02 | 10:00 | the computer moves the string |
| 3 · From Neuron to Network | D01.03 | 11:15 | a neuron, then a network that sees |
| 4 · How an LLM Reads | D01.04 | 1:00 | tokens, next words, meaning as numbers |
| 5 · Build a RAG | D01.05 | 2:00 | an AI that answers from your documents |
| 6 · Break It, Then Ship It | D01.06 | 3:15 | test it, find what fails, make it yours |

**Open the table of contents** (the ☰ icon on the left) to jump between parts.

**How to use it**
- **Run one part at a time, top to bottom,** with Shift-Enter (or each cell's ▶ button), reading
  as you go. *Runtime → Run all* also works — it runs every part, just slower.
- **If Colab disconnects** (it will over lunch): re-run the ⚙️ SETUP cell of the part you're in.
- **Tinker** — move the sliders and change the boxes on the right of the ✍️ and 🔍 cells.
- **Build** — Parts 2, 4 and 6 end with a ⭐ section for anyone who finishes early.

| Marker | Means |
| --- | --- |
| ▶ DEMO | just run it |
| 🔍 READ | **predict first**, write it down, then run |
| ✍️ YOU TRY | your turn to change something |
| ✅ CHECKPOINT | tells you whether you got it |

Every cell is numbered **part.section.cell** — so *✅ 2.2.2* is Part 2, section 2, the second cell.
Stuck? Red sticky note on your lid, or select a cell and ask the AI assistant to explain it.

# Part 1 · Get Ready
**Deck D01.01** · do this **the day before class**, then check it again at 9:20

1. **File → Save a copy in Drive.** Use *your copy* all day — it keeps your changes. (In class:
   *File → Open notebook → Recent*.)
2. Make sure you are **signed in to Google** (top right).
3. Run the **three cells in this part** with the ▶ button on each (or Shift-Enter). The second one
   takes about a minute.
4. When the last one says **✅ READY FOR CLASS**, take a **screenshot** and send it to your instructor.

**Something red?** Screenshot the error instead and send that. The most common cause is a work
laptop that blocks Google services; try a personal laptop or browser profile.

## 1.1 · Check your laptop

In [ ]:
# ▶ DEMO 1.1.1 — the basics
import platform
import numpy, matplotlib, sklearn, ipywidgets

print("Python    ", platform.python_version())
print("numpy     ", numpy.__version__)
print("matplotlib", matplotlib.__version__)
print("sklearn   ", sklearn.__version__)
print("✅ The basics are installed.")

In [ ]:
# ▶ DEMO 1.1.2 — the tools for the afternoon: an embedding model and a PDF reader (about a minute)
import importlib.util, subprocess, sys

for package, module in (("sentence-transformers", "sentence_transformers"), ("pypdf", "pypdf")):
    if importlib.util.find_spec(module) is None:
        print(f"installing {package} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

from sentence_transformers import SentenceTransformer
import pypdf

_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
_a, _b, _c = _model.encode(["cut carbon emissions", "reduce CO2 output", "quarterly earnings call"],
                           normalize_embeddings=True)
close, far = float(_a @ _b), float(_a @ _c)
print(f"'cut carbon emissions' vs 'reduce CO2 output':       {close:.2f}")
print(f"'cut carbon emissions' vs 'quarterly earnings call': {far:.2f}")
if close > far:
    print("✅ The embedding model works: it knows those first two mean nearly the same thing.")
else:
    print("❌ The model loaded but gave odd results — screenshot this for your instructor.")

In [ ]:
# ▶ DEMO 1.1.3 — all done?
print("✅ READY FOR CLASS — take a screenshot of this and send it in.")

# Part 2 · Train Your First Model
**Deck D01.02** · 10:00

The ice-cream data from your wall plot. This time the computer moves the string.

In [ ]:
# @title ⚙️ PART 2 SETUP — run this first, and again if Colab reconnects { display-mode: "form" }
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from ipywidgets import interact, FloatSlider, fixed

temp_f = np.array([58, 62, 64, 67, 70, 72, 75, 77, 80, 84, 88, 93])   # afternoon high, °F
cones  = np.array([95, 160, 150, 210, 230, 290, 310, 355, 340, 430, 470, 540])   # cones sold that day


def total_squares(m, b):
    """The score from the wall: square every gap, add them up."""
    gaps = cones - (m * temp_f + b)
    return float((gaps ** 2).sum())


def draw_line(m, b, title=""):
    """The data, a line, and every gap drawn as a gold square."""
    fig, ax = plt.subplots(figsize=(8, 5))
    xs = np.array([50, 100])
    ax.plot(xs, m * xs + b, color="crimson", lw=3)
    for x, y in zip(temp_f, cones):
        guess = m * x + b
        gap = abs(y - guess)
        ax.add_patch(Rectangle((x, min(y, guess)), gap * (50 / 650) * (5 / 8), gap,
                               color="gold", alpha=0.45, lw=0))
        ax.plot([x, x], [y, guess], color="black", lw=1)
    ax.scatter(temp_f, cones, s=70, zorder=3)
    ax.set_xlim(50, 100); ax.set_ylim(0, 650)
    ax.set_xlabel("Afternoon high (°F)"); ax.set_ylabel("Cones sold")
    ax.set_title(f"{title}   total of the squares = {total_squares(m, b):,.0f}")
    plt.show()


def train(lr, steps=100):
    """Gradient descent from a flat line at zero. Returns m, b and the total after every step.

    Behind the scenes the temperatures are rescaled so that ONE learning rate suits both
    numbers; m and b are converted back to cones-per-degree before they are returned.
    """
    mx, sx = temp_f.mean(), temp_f.std()
    xs = (temp_f - mx) / sx
    w = c = 0.0
    history = []
    for _ in range(steps):
        err = (w * xs + c) - cones
        history.append(float((err ** 2).sum()))
        if history[-1] > 1e12:            # it blew up, stop before the numbers overflow
            break
        w -= lr * 2 * np.mean(err * xs)   # step downhill for the slope...
        c -= lr * 2 * np.mean(err)        # ...and for the intercept
    history.append(float((((w * xs + c) - cones) ** 2).sum()))
    return w / sx, c - w * mx / sx, history


def show_training(lr, moments):
    """The same training run, frozen at a few moments, plus the loss curve."""
    fig, axes = plt.subplots(1, len(moments) + 1, figsize=(4 * (len(moments) + 1), 3.6))
    for ax, k in zip(axes, moments):
        m, b, h = train(lr, steps=k)
        ax.scatter(temp_f, cones, s=25)
        ax.plot([50, 100], [m * 50 + b, m * 100 + b], color="crimson", lw=2.5)
        ax.set_ylim(-50, 650); ax.set_title(f"after {k} steps\ntotal {h[-1]:,.0f}", fontsize=10)
    _, _, h = train(lr, steps=max(moments))
    axes[-1].semilogy(h, lw=2.5); axes[-1].set_title("total of the squares\n(log scale)", fontsize=10)
    axes[-1].set_xlabel("step")
    plt.tight_layout(); plt.show()


def compare_rates(rates, steps=60):
    """Several learning rates side by side: the slope it has found, and the loss."""
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.5))
    for lr in rates:
        slopes = [train(lr, steps=k)[0] for k in range(steps + 1)]
        _, _, h = train(lr, steps=steps)
        a1.plot(slopes, lw=2, label=f"learning rate {lr}")
        a2.semilogy(h, lw=2, label=f"learning rate {lr}")
    a1.axhline(np.polyfit(temp_f, cones, 1)[0], color="black", ls="--", lw=1, label="best slope")
    a1.set_ylim(-5, 30); a1.set_title("the slope it has found"); a1.set_xlabel("step")
    a2.set_title("total of the squares (log scale)"); a2.set_xlabel("step")
    a1.legend(fontsize=9); plt.tight_layout(); plt.show()


def show_prediction(temperature, m, b):
    fig, ax = plt.subplots(figsize=(9, 5))
    xs = np.array([0, 120])
    ax.axvspan(temp_f.min(), temp_f.max(), color="green", alpha=0.08, label="where we have data")
    ax.plot(xs, m * xs + b, color="crimson", lw=2.5)
    ax.axhline(0, color="black", lw=0.8)
    ax.scatter(temp_f, cones, s=40, zorder=3)
    ax.scatter([temperature], [m * temperature + b], s=200, marker="*", color="black", zorder=4,
               label=f"prediction at {temperature}°F")
    ax.set_xlabel("Afternoon high (°F)"); ax.set_ylabel("Cones sold"); ax.legend(loc="upper left")
    plt.show()


def line_str(m, b):
    """cones = 12.62 × temp − 637.6"""
    return f"cones = {m:.2f} × temp {'−' if b < 0 else '+'} {abs(b):.1f}"


def check(ok, good, bad):
    print(("✅ " + good) if ok else ("❌ " + bad))


print("✅ Setup done — 12 days of ice-cream data loaded.")

## 2.1 · The data

In [ ]:
# ▶ DEMO 2.1.1 — the same twelve days as your wall plot
for t, c in zip(temp_f, cones):
    print(f"{t}°F  →  {c} cones")

plt.scatter(temp_f, cones, s=80)
plt.xlabel("Afternoon high (°F)"); plt.ylabel("Cones sold"); plt.title("Ice cream sales — 12 days")
plt.show()

## 2.2 · Be the string, again

Same game as the wall. A line is just **two numbers**:

- **m**, the slope: how many extra cones for each extra degree
- **b**, the intercept: where the line crosses zero degrees

Move them and watch the score, the **total of the squares**. Lower is better.

In [ ]:
# ✍️ YOU TRY 2.2.1 — drag the sliders. Can you get the total under 5,000?
interact(draw_line,
         m=FloatSlider(value=10, min=0, max=20, step=0.1, description="slope m", continuous_update=False),
         b=FloatSlider(value=-500, min=-1000, max=200, step=10, description="intercept b", continuous_update=False),
         title=fixed("Your line"));

In [ ]:
# ✅ CHECKPOINT 2.2.2 — type your best slope and intercept into the boxes, then run this cell
my_m = 10.0   # @param {type:"number"}
my_b = -500.0 # @param {type:"number"}

score = total_squares(my_m, my_b)
print(f"Your line: {line_str(my_m, my_b)}   →   total of the squares = {score:,.0f}")
check(score < 5000, "Under 5,000. You just trained a model by hand.",
      "Not under 5,000 yet. Back to the sliders: when you change the slope, fix the intercept too.")

**That's all a model is here: two numbers.** *Training* means finding good values for them.

## 2.3 · Let the computer move the string

You moved the line by eye. The computer runs the same loop — **guess → measure → adjust** —
but it works out exactly **which way is downhill** for each number, and takes a small step that
way. That is **gradient descent**. The size of each step is the **learning rate**.

It starts from the worst possible guess: a flat line at zero cones.

In [ ]:
# 🔍 READ 2.3.1 — predict first: after 100 steps, will the computer beat your best score?
# My prediction:

m, b, history = train(lr=0.1, steps=100)
print(f"After 100 steps: {line_str(m, b)}")
print(f"Total of the squares: {history[-1]:,.0f}   (it started at {history[0]:,.0f})")
draw_line(m, b, "The computer's line after 100 steps")

In [ ]:
# ▶ DEMO 2.3.2 — the same training, frozen at five moments
show_training(lr=0.1, moments=[0, 2, 5, 10, 30])

## 2.4 · The learning rate

The learning rate is **how hard you twist the string** on each try. Too gentle and it takes
forever; too hard and it overshoots. One picture says it all:

In [ ]:
# ▶ DEMO 2.4.1 — five learning rates, side by side
compare_rates([0.001, 0.01, 0.1, 0.85, 1.1])

**Crawls** (0.001) · **smooth** (0.1) · **zig-zags** (0.85) · **blows up** (1.1). That's all you need to
know about it today: it's a knob engineers tune, and the loop only works if it's set sensibly.

## 2.5 · The one-liner

For a straight line there's a formula, so in practice nobody runs the loop. This is what people
actually type:

In [ ]:
# ▶ DEMO 2.5.1 — the formula, and the loop, side by side
m_fit, b_fit = np.polyfit(temp_f, cones, 1)    # 1 = a straight line
m_gd, b_gd, _ = train(lr=0.1, steps=500)

print(f"formula (polyfit):  {line_str(m_fit, b_fit)}")
print(f"gradient descent:   {line_str(m_gd, b_gd)}")
check(abs(m_fit - m_gd) < 0.01, "Same line, two ways.", "They disagree. Ask for help, this should not happen.")

**Why learn the loop, then?** Because a neural network has no formula. The loop is the only way
to train one, and that's the next hour.

## 2.6 · When the line lies

In [ ]:
# 🔍 READ 2.6.1 — predict first: how many cones on a freezing 32°F day? Then move the slider.
degrees = 32  # @param {type:"slider", min:0, max:120, step:1}

print(f"At {degrees}°F the line predicts {m_fit * degrees + b_fit:,.0f} cones.")
show_prediction(degrees, m_fit, b_fit)

At 32°F the line predicts **about −230 cones**. Nobody sells negative ice cream.

**Talk at your table:**
1. Between which temperatures would you trust this line? Why there?
2. What does this model **not know**? (Think: weekends, rain, holidays, the shop running out.)
3. Remember the circle puzzle. Where did that pattern break, and what's the same here?

## ⭐ 2.7 · Build (optional)

In [ ]:
# ✍️ YOU TRY 2.7.1 — one bad day. On a 100°F day the freezer broke and you sold 40 cones.
# Add that day, refit the line, and compare.
temp_bad  = np.append(temp_f, 100)
cones_bad = np.append(cones, 40)

m_bad, b_bad = 0.0, 0.0      # TODO: replace with np.polyfit(temp_bad, cones_bad, 1)

print(f"Without the bad day: {m_fit:.2f} cones per °F")
print(f"With the bad day:    {m_bad:.2f} cones per °F")

Squaring the gaps is what makes one big miss so loud. Is that what you want? It depends on
whether the bad day is a mistake in the data or a real risk you need to plan for.

In [ ]:
# ✍️ YOU TRY 2.7.2 — too little data. Train on just two days, 77°F and 80°F, then predict 93°F.
two = (temp_f == 77) | (temp_f == 80)
m_two, b_two = np.polyfit(temp_f[two], cones[two], 1)
print(f"Trained on two days: {m_two:+.1f} cones per °F")

predicted_93 = 0.0      # TODO: use m_two and b_two to predict sales at 93°F

print(f"At 93°F it predicts {predicted_93:,.0f} cones. The real answer was 540.")

Two days, and the model is sure that **hotter days sell fewer cones**. Every model is only as good
as the examples it learned from — remember that when we get to LLMs after lunch.

## Part 2 wrap

You can now:
- say what **training** means — finding the numbers that make the loss smallest
- name the **loss**, **gradient descent** and the **learning rate**, and say what each does
- say where a model **stops being trustworthy**

**Next — D01.03, From Neuron to Network:** the same loop, for thousands of numbers at once.

# Part 3 · From Neuron to Network
**Deck D01.03** · 11:15

One artificial neuron, then a small network that learns to see the Glimmerscope's 2×2 pictures.

In [ ]:
# @title ⚙️ PART 3 SETUP — run this first, and again if Colab reconnects { display-mode: "form" }
import warnings
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
warnings.filterwarnings("ignore")

CATEGORIES = ["solid", "vertical", "horizontal", "diagonal"]


def make_images(n, noise=0.08, seed=0):
    """n random 2×2 pictures. Pixels run from -1 (white) to +1 (black), in the order
    top-left, top-right, bottom-left, bottom-right."""
    rng = np.random.default_rng(seed)
    X, y = [], []
    for _ in range(n):
        kind = rng.integers(4)
        a = rng.uniform(-1, 1)
        b = a
        while kind != 0 and abs(a - b) < 0.8:      # two clearly different shades
            b = rng.uniform(-1, 1)
        tl, tr, bl, br = {0: (a, a, a, a), 1: (a, b, a, b),
                          2: (a, a, b, b), 3: (a, b, b, a)}[kind]
        X.append(np.clip(np.array([tl, tr, bl, br]) + rng.normal(0, noise, 4), -1, 1))
        y.append(kind)
    return np.array(X), np.array(y)


def _frame(ax):
    """No ticks, but a visible border, so white pixels don't vanish into the page."""
    ax.set_xticks([]); ax.set_yticks([])
    for side in ax.spines.values():
        side.set_visible(True); side.set_color("gray")


def show_images(X, y, predicted=None, per_row=8):
    n = len(X)
    rows = (n + per_row - 1) // per_row
    fig, axes = plt.subplots(rows, per_row, figsize=(1.6 * per_row, 1.9 * rows))
    for ax, i in zip(np.ravel(axes), range(rows * per_row)):
        ax.axis("off")
        if i >= n:
            continue
        ax.imshow(X[i].reshape(2, 2), cmap="gray_r", vmin=-1, vmax=1)
        _frame(ax)
        label = CATEGORIES[y[i]]
        if predicted is not None and predicted[i] != y[i]:
            ax.set_title(f"✗ {CATEGORIES[predicted[i]]}", color="crimson", fontsize=10)
        else:
            ax.set_title(label, fontsize=10)
    plt.tight_layout(); plt.show()


def show_guess(net, pixels):
    probs = net.predict_proba([pixels])[0]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3), gridspec_kw={"width_ratios": [1, 2.5]})
    a1.imshow(np.array(pixels).reshape(2, 2), cmap="gray_r", vmin=-1, vmax=1); _frame(a1)
    a1.set_title("your picture")
    a2.barh(CATEGORIES, probs, color=["crimson" if p == probs.max() else "lightgray" for p in probs])
    a2.set_xlim(0, 1); a2.set_title("how sure the network is")
    plt.tight_layout(); plt.show()


def check(ok, good, bad):
    print(("✅ " + good) if ok else ("❌ " + bad))


X_train, y_train = make_images(400, seed=1)
X_test, y_test = make_images(200, seed=2)      # pictures the network never sees while learning
print("✅ Setup done — 400 practice pictures and 200 test pictures made.")

## 3.1 · One neuron

Frank Rosenblatt's **perceptron** (1957): multiply each input by a weight, add them up, and
**fire** (answer 1) if the total reaches a threshold. That's the whole thing.

In [ ]:
# ▶ DEMO 3.1.1 — a perceptron, in four lines
def perceptron(inputs, weights, threshold):
    weighted_sum = sum(x * w for x, w in zip(inputs, weights))
    return 1 if weighted_sum >= threshold else 0

# Hand-picked weights that make it an OR gate: fire if EITHER input is on
for a, b in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    print(f"{a} OR {b}  →  {perceptron([a, b], weights=[1, 1], threshold=0.5)}")

In [ ]:
# ✅ CHECKPOINT 3.1.2 — make it an AND gate: fire only if BOTH inputs are on.
# Change the numbers in the boxes, then run.
w1 = 1.0          # @param {type:"number"}
w2 = 1.0          # @param {type:"number"}
threshold = 0.5   # @param {type:"number"}

AND = {(0, 0): 0, (0, 1): 0, (1, 0): 0, (1, 1): 1}
right = 0
for (a, b), want in AND.items():
    got = perceptron([a, b], [w1, w2], threshold)
    right += got == want
    print(f"{a} AND {b}  →  {got}   {'✓' if got == want else '✗ should be ' + str(want)}")
check(right == 4, "That's an AND gate.", "Not yet. Which number decides how much input it takes to fire?")

## 3.2 · Let it learn

You picked the weights by hand. Now the neuron finds them itself — with **the loop**:
guess, see how wrong it was, nudge each weight a little in the direction that would have helped.

In [ ]:
# ▶ DEMO 3.2.1 — the learning rule. Same loop as the string: guess → how wrong → adjust.
def train_perceptron(table, learning_rate=0.1, passes=20):
    weights, threshold = [0.0, 0.0], 0.0
    for p in range(1, passes + 1):
        mistakes = 0
        for inputs, target in table.items():
            error = target - perceptron(inputs, weights, threshold)   # +1, 0 or -1
            if error:
                mistakes += 1
                weights = [w + learning_rate * error * x for w, x in zip(weights, inputs)]
                threshold -= learning_rate * error
        if p <= 5 or mistakes == 0 or p == passes:
            print(f"pass {p:2}: {mistakes} mistake(s)   weights = {weights[0]:.1f}, {weights[1]:.1f}   threshold = {threshold:.1f}")
        elif p == 6:
            print("   ...")
        if mistakes == 0:
            print("✅ learned it")
            return
    print("❌ gave up — still making mistakes")

In [ ]:
# 🔍 READ 3.2.2 — predict first: how many passes through the table to learn AND?
train_perceptron({(0, 0): 0, (0, 1): 0, (1, 0): 0, (1, 1): 1})

In [ ]:
# 🔍 READ 3.2.3 — XOR: fire if EXACTLY ONE input is on. Predict: can one neuron learn it?
train_perceptron({(0, 0): 0, (0, 1): 1, (1, 0): 1, (1, 1): 0})

**One neuron draws one straight line** between "fire" and "don't fire". For XOR, no single
straight line separates the 1s from the 0s — try drawing it on paper.

In 1969, Minsky and Papert's book *Perceptrons* made this limit famous, and interest in neural
networks collapsed for more than a decade. The fix turned out to be **layers** of neurons, and a
way to train all of them at once (**backpropagation**, popularised in 1986). That's what we
use next.

## 3.3 · The Glimmerscope

The Glimmerscope sees **2×2 pictures**. Each pixel is a number from −1 (white) to +1 (black).
The job: say whether a picture is **solid**, **vertical**, **horizontal** or **diagonal**.

In [ ]:
# ▶ DEMO 3.3.1 — sixteen practice pictures, with their right answers
show_images(X_train[:16], y_train[:16])

In [ ]:
# ▶ DEMO 3.3.2 — a small network: 4 pixels in → 8 hidden neurons → 4 answers out.
# It trains with the same loop; the "loss curve" shows it getting less wrong.
net = MLPClassifier(hidden_layer_sizes=(8,), activation="relu", max_iter=3000, random_state=0)
net.fit(X_train, y_train)

print(f"right on the practice pictures: {net.score(X_train, y_train):.0%}")
print(f"right on NEW pictures:          {net.score(X_test, y_test):.0%}")

plt.plot(net.loss_curve_, lw=2.5)
plt.xlabel("pass through the practice pictures"); plt.ylabel("loss (how wrong)")
plt.title("the loop, again"); plt.show()

In [ ]:
# ▶ DEMO 3.3.3 — new pictures it never saw. Any it gets wrong are marked ✗ in red.
show_images(X_test[:24], y_test[:24], predicted=net.predict(X_test[:24]))

In [ ]:
# 🔍 READ 3.3.4 — how many hidden neurons does it need? Predict, then try each value in the box.
hidden_neurons = 1  # @param [1, 2, 4, 8, 16] {type:"raw"}

small = MLPClassifier(hidden_layer_sizes=(hidden_neurons,), activation="relu", max_iter=3000, random_state=0)
small.fit(X_train, y_train)
print(f"{hidden_neurons} hidden neuron(s): right on new pictures {small.score(X_test, y_test):.0%}")

In [ ]:
# ✍️ YOU TRY 3.3.5 — draw your own picture with the sliders (-1 white … +1 black) and see what it says.
# Try to fool it: what about a picture that's almost solid?
top_left     = 1.0   # @param {type:"slider", min:-1, max:1, step:0.1}
top_right    = 1.0   # @param {type:"slider", min:-1, max:1, step:0.1}
bottom_left  = -1.0  # @param {type:"slider", min:-1, max:1, step:0.1}
bottom_right = -1.0  # @param {type:"slider", min:-1, max:1, step:0.1}

show_guess(net, [top_left, top_right, bottom_left, bottom_right])

## 3.4 · Too little to learn from: overfitting

In [ ]:
# ✍️ YOU TRY 3.4.1 — a big network with only a few practice pictures. Then fix it.
practice_pictures = 8   # @param {type:"integer"}

big = MLPClassifier(hidden_layer_sizes=(64,), activation="relu", max_iter=3000, random_state=0)
big.fit(X_train[:practice_pictures], y_train[:practice_pictures])
on_practice, on_new = big.score(X_train[:practice_pictures], y_train[:practice_pictures]), big.score(X_test, y_test)
print(f"practice pictures: {on_practice:.0%}   new pictures: {on_new:.0%}")
check(on_new >= 0.9, "Now it generalises.",
      "It memorised the practice pictures but can't handle new ones — that's overfitting. Give it more pictures.")

## Part 3 wrap

You can now:
- say what one **neuron** does — weigh the inputs, add, fire or not
- say why one neuron isn't enough (**XOR**) and what layers add
- train a network and test it on pictures it **never saw** — the only score that counts
- spot **overfitting**: perfect on practice, poor on anything new

**Next — D01.04, How an LLM Reads:** the same networks, scaled up to read text.

# Part 4 · How an LLM Reads
**Deck D01.04** · 1:00

Three things every LLM does: chop text into **tokens**, predict the **next token**, and turn meaning into numbers (**embeddings**).

In [ ]:
# @title ⚙️ PART 4 SETUP — run this first, and again if Colab reconnects (about a minute) { display-mode: "form" }
import importlib.util, subprocess, sys, warnings
warnings.filterwarnings("ignore")
if importlib.util.find_spec("sentence_transformers") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=True)

import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

# A small, old LLM (2019, 82 million numbers). Today's chat models are thousands of times bigger,
# but they read and predict in exactly this way.
tok = AutoTokenizer.from_pretrained("distilgpt2")
llm = AutoModelForCausalLM.from_pretrained("distilgpt2").eval()

# An embedding model: turns any sentence into 384 numbers that capture its meaning.
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def show_tokens(text):
    ids = tok.encode(text)
    pieces = [tok.decode([i]) for i in ids]
    print(f"{len(pieces)} tokens:  " + " ".join(f"[{p}]" for p in pieces))


def next_words(text, k=10):
    """The model's top-k guesses for the next token, with probabilities."""
    with torch.no_grad():
        logits = llm(**tok(text, return_tensors="pt")).logits[0, -1]
    probs = torch.softmax(logits, dim=-1)
    top = torch.topk(probs, k)
    words = [tok.decode([int(i)]) for i in top.indices]
    return words, [float(p) for p in top.values]


def show_next(text, k=10):
    words, probs = next_words(text, k)
    plt.figure(figsize=(8, 0.4 * k + 1))
    plt.barh([repr(w) for w in words][::-1], probs[::-1], color="steelblue")
    plt.xlabel("probability"); plt.title(f"after: \"{text}\"", fontsize=11)
    plt.show()


def generate(text, words=20, temperature=0.7, seed=0):
    """Pick one token at a time. Low temperature: always the likeliest. High: more random."""
    torch.manual_seed(seed)
    ids = tok.encode(text, return_tensors="pt")
    for _ in range(words):
        with torch.no_grad():
            logits = llm(ids).logits[0, -1] / temperature
        nxt = torch.multinomial(torch.softmax(logits, dim=-1), 1)
        ids = torch.cat([ids, nxt.view(1, 1)], dim=1)
    return tok.decode(ids[0])


def similarity(texts):
    v = embedder.encode(texts, normalize_embeddings=True)
    return v @ v.T


def heatmap(texts):
    s = similarity(texts)
    fig, ax = plt.subplots(figsize=(10, 8.5))
    im = ax.imshow(s, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(texts))); ax.set_xticklabels(texts, rotation=60, ha="right")
    ax.set_yticks(range(len(texts))); ax.set_yticklabels(texts)
    for i in range(len(texts)):
        for j in range(len(texts)):
            ax.text(j, i, f"{s[i, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if s[i, j] > 0.6 else "black")
    plt.colorbar(im, label="how alike (1 = same meaning)"); plt.tight_layout(); plt.show()


def meaning_map(texts, highlight=()):
    """Squash 384 numbers down to 2 so we can look at them (PCA)."""
    v = embedder.encode(texts, normalize_embeddings=True)
    v = v - v.mean(axis=0)
    xy = v @ np.linalg.svd(v, full_matrices=False)[2][:2].T
    plt.figure(figsize=(10, 7))
    for k, ((x, y), t) in enumerate(zip(xy, texts)):
        mine = t in highlight
        plt.scatter(x, y, s=120 if mine else 60, color="crimson" if mine else "steelblue")
        plt.annotate(t, (x, y), xytext=(6, 4 if k % 2 else -12), textcoords="offset points",
                     fontsize=11, weight="bold" if mine else "normal")
    plt.xticks([]); plt.yticks([]); plt.title("the meaning map (384 numbers squashed to 2)")
    plt.show()


def check(ok, good, bad):
    print(("✅ " + good) if ok else ("❌ " + bad))


print("✅ Setup done — an LLM and an embedding model are loaded.")

## 4.1 · Tokens

An LLM never sees letters or words. It sees **tokens** — common chunks of text, each with an ID
number. Common words are one token; rare words get split up.

In [ ]:
# ▶ DEMO 4.1.1 — how text gets chopped up
for text in ["The ice cream sold out.",
             "unbelievably",
             "decarbonisation",
             "https://www.ucsd.edu",
             "Guten Morgen, wie geht's?"]:
    show_tokens(text)

In [ ]:
# ✍️ YOU TRY 4.1.2 — your name, your company, a word from your field. How many tokens?
my_text = "Convergent Systems Engineering"  # @param {type:"string"}
show_tokens(my_text)

Rare words, names and other languages cost **more tokens**. That matters: LLMs are priced and
limited by tokens, not words.

## 4.2 · Predict the next token

That's the whole job of an LLM during training: read some text, guess the next token, see how
wrong it was, adjust. **The loop**, on most of the internet.

In [ ]:
# 🔍 READ 4.2.1 — predict first: what's the most likely next word? Write it down, then run.
show_next("The ice cream shop sold out because it was very")

In [ ]:
# ✍️ YOU TRY 4.2.2 — your own unfinished sentence
my_sentence = "To reduce our carbon emissions, the company will"  # @param {type:"string"}
show_next(my_sentence)

In [ ]:
# 🔍 READ 4.2.3 — generating = predicting, picking one, repeating. Try each temperature.
# Predict: what will a high temperature do to the text?
temperature = 0.7  # @param [0.1, 0.7, 1.5] {type:"raw"}
print(generate("The best way to save energy at home is", words=25, temperature=temperature))

- **Low temperature** — always picks the likeliest token: safe, repetitive.
- **High temperature** — picks less likely tokens more often: varied, and soon nonsense.

This is a small, old model, so even its best text wanders. Bigger models learned the same loop on
far more text — but they are still **predicting likely text, not looking up facts**. Keep that in
mind this afternoon.

## 4.3 · Embeddings: meaning as numbers

An embedding model turns any sentence into **384 numbers**. Sentences that mean similar things
get similar numbers — even when they share no words. These are the same twelve cards you sorted
on your desk.

In [ ]:
# ▶ DEMO 4.3.1 — the twelve cards
cards = ["cut carbon emissions", "reduce CO2 output", "lower our carbon footprint",
         "save water", "reduce water use", "fix leaking pipes",
         "quarterly earnings", "profit margin", "revenue growth",
         "recycle plastic", "compost food scraps",
         "send a carbon copy"]
print(len(cards), "cards")

In [ ]:
# 🔍 READ 4.3.2 — predict first: which card is most like "cut carbon emissions"?
# And where will "send a carbon copy" end up — near the emissions cards, or not?
heatmap(cards)

In [ ]:
# ▶ DEMO 4.3.3 — the same 12 cards as a map: close together = similar meaning
meaning_map(cards)

In [ ]:
# ✅ CHECKPOINT 4.3.4 — which card does the model think is closest to "cut carbon emissions"?
# Type it exactly as it appears in the list, then run.
my_answer = "send a carbon copy"  # @param {type:"string"}

s = similarity(cards)[0]
best = cards[int(np.argsort(s)[-2])]          # [-1] is the card itself
check(my_answer == best, f"Right: '{best}' ({s[cards.index(best)]:.2f}).",
      "Not that one. Look along the top row of the heatmap.")

In [ ]:
# ✍️ YOU TRY 4.3.5 — add three sentences of your own. Where do they land?
mine = ["install solar panels",   # @param {type:"string"}
        "shorter showers",        # @param {type:"string"}
        "beat analyst forecasts"] # @param {type:"string"}
meaning_map(cards + mine, highlight=mine)

Two things to look for in the heatmap:
- **"send a carbon copy"** has nothing to do with emissions, but shares the word *carbon* — and
  this small model scores it closer to "cut carbon emissions" than "save water" is. Meaning
  models still lean on words.
- **"fix leaking pipes"** — most desks put it with the water cards. Does the model?

**Talk at your table:** where did the model disagree with your desk sort? Who decided what
"similar" means — and what text did the model learn that from?

## ⭐ 4.4 · Build (optional): search by meaning

This is the heart of this afternoon's RAG. Search **by meaning** finds the right sentence even when
it shares no words with the question.

In [ ]:
# ▶ DEMO 4.4.1 — a tiny library, and two ways to search it
library = [
    "Our freezers now run on electricity from rooftop solar panels.",
    "We switched delivery vans to electric, cutting fuel use by half.",
    "Cones are made with flour from farms within 100 miles.",
    "Leftover waffle batter is composted at a community garden.",
    "Paper cups replaced plastic ones in 2024.",
    "Staff get two paid volunteer days a year.",
    "Sales rose 12% in the summer quarter.",
    "Water use per scoop fell after we fixed the dipper-well taps.",
]
library_vectors = embedder.encode(library, normalize_embeddings=True)

def by_meaning(question, k=3):
    scores = library_vectors @ embedder.encode([question], normalize_embeddings=True)[0]
    return [(float(scores[i]), library[i]) for i in np.argsort(scores)[::-1][:k]]

def by_keywords(question, k=3):
    q = set(question.lower().replace("?", "").split())
    scores = [len(q & set(s.lower().replace(".", "").split())) for s in library]
    return [(scores[i], library[i]) for i in np.argsort(scores)[::-1][:k]]

In [ ]:
# ✍️ YOU TRY 4.4.2 — ask a question. Try one that shares NO words with the right answer.
question = "How did they lower their carbon footprint from transport?"  # @param {type:"string"}

print("BY KEYWORDS")
for score, s in by_keywords(question):
    print(f"  {score} shared words   {s}")
print("\nBY MEANING")
for score, s in by_meaning(question):
    print(f"  {score:.2f}   {s}")

## Part 4 wrap

You can now:
- say what a **token** is, and why rare words cost more
- say what an LLM actually does: **predict the next token**, again and again
- say what **temperature** changes
- say what an **embedding** is: meaning as numbers, so similar ideas sit close together

**Next — D01.05, Build a RAG:** search by meaning, plus next-token prediction, on your own documents.

# Part 5 · Build a RAG
**Deck D01.05** · 2:00

**RAG** — *retrieval-augmented generation* — means: find the passages in **your** documents that
match a question, then hand them to an AI along with the question, so it answers from your
documents instead of from memory.

You'll build every step and **see what each one produces**. The last step — an AI writing the
answer — uses **Colab's built-in AI** if your Google account has it. If it doesn't, the notebook
says so, and you paste the prompt into **your own AI** (ChatGPT, Claude, Gemini, Copilot…) instead.
No API key either way.

**Your documents:** drag your team's PDFs into the **📁 Files** panel on the left (the folder icon),
then run this part. No PDFs? It uses a built-in sample report instead.

⚠️ **Public documents only.** No CUI, no ITAR/EAR technical data, nothing classified, nothing from a
program, no personnel records — this part sends text to an AI. If you're not sure it's public, it isn't.

In [ ]:
# @title ⚙️ PART 5 SETUP — run this first, and again if Colab reconnects (about a minute) { display-mode: "form" }
import glob, html, importlib.util, subprocess, sys, textwrap, warnings
warnings.filterwarnings("ignore")
for package, module in (("sentence-transformers", "sentence_transformers"), ("pypdf", "pypdf")):
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import numpy as np
import matplotlib.pyplot as plt
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from IPython.display import HTML, display

IN_COLAB = "google.colab" in sys.modules
if "embedder" not in globals():      # Part 4 already loaded it, unless Colab reconnected
    embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

SAMPLE_NAME = "KESTREL lessons-learned report (fictional sample)"
SAMPLE = """=== page 1 ===

Program KESTREL — Lessons Learned Report, Phase B
Coastal Systems Integration Center (fictional) · Unclassified sample document

Executive summary

KESTREL integrates a new sensor and communications package onto an existing fleet of uncrewed
surface vessels. Phase B covered detailed design through first article test. The package met its
performance requirements, but the phase finished late and over budget, and most of the cost came
from problems we had seen before on earlier programs.

In short: the schedule slipped four months, rework cost $2.1 million, and three of the five
lessons below were already in our own lessons-learned database from earlier programs. We wrote
them down. We did not learn them.

This report is written for the next program team, not for the record. Each lesson ends with one
recommendation we believe would have prevented it.

How we collected these lessons

We held eleven structured interviews with the integrated product team, reviewed 214 change
requests, and compared our plan against what happened at each review. Interviews were
confidential; no names appear in this report.

=== page 2 ===

Lesson 1 — Requirements changed, and the tests did not

After the critical design review, 37 requirements changed. Nine of those changes were never
carried through to the test procedures, because the link between requirements and tests was kept
in a spreadsheet that was updated by hand before each review. At first article test, two of the
nine caused failures that took eleven weeks to trace back to their cause.

Recommendation: keep requirement-to-test links in the requirements tool itself, and treat any
requirement change without an updated test link as an open action item.

Lesson 2 — The interface lived in one person's head

The legacy vessel's power-and-data interface was documented in 2009 and never updated. The only
engineer who understood its undocumented behaviour retired in the second month of Phase B. Two
contractors spent seven weeks reverse-engineering what one conversation could have told them.

Recommendation: before a key engineer leaves, schedule recorded walkthroughs of anything they own
that is not fully documented, and budget the time for it.

Lesson 3 — A part went end-of-life mid-design

The processing board's main programmable chip was announced end-of-life four months into Phase B.
We made a lifetime buy of 1,200 units, enough for the planned fleet plus 15% spares, at a cost of
$640,000. A redesign for a newer chip is planned for a later phase.

Recommendation: check the obsolescence status of every critical part at each design review, not
only at the start of the program.

=== page 3 ===

Lesson 4 — The AI assistant was right in the demo

We piloted an AI assistant that summarises change requests for engineers. In the demonstration it
was accurate on every example. In daily use it summarised against an out-of-date copy of the
requirements database, and twice gave confident answers based on requirements that had already
been changed. No harm resulted, because an engineer checked each summary before acting on it.

Recommendation: continue the pilot only with a human approving every summary, and with a check
that the assistant is reading the current database before each use.

Lesson 5 — A marking mistake on a shared drawing set

A drawing set was shared with a subcontractor before its controlled-information markings had
been reviewed. The error was caught within a day and corrected, and no information left approved
systems. It still cost a week of reviews and a formal report.

Recommendation: review markings before any document leaves the team, not after.

Timeline

| Milestone | Planned | Actual |
| --- | --- | --- |
| Critical design review | Month 3 | Month 3 |
| Test procedures complete | Month 7 | Month 10 |
| First article test | Month 9 | Month 15 |

Measured from the critical design review to first article test, Phase B finished six months late.

=== page 4 ===

What we would do differently

Most of our problems were not technical. They came from defaults nobody questioned: that a
spreadsheet would stay up to date, that the documents held what our people knew, that a part
available at the start would stay available, and that a tool that worked in a demonstration would
work in daily use.

Next steps for Phase C

- Move requirement-to-test traceability into the requirements tool before Phase C begins
- Record walkthroughs of the three remaining interfaces that depend on a single engineer
- Add an obsolescence check to every design review checklist
- Re-run the AI assistant pilot for three months, with the safeguards in Lesson 4
- Add a markings review to the document-release checklist"""


def load_documents():
    """Every PDF in the Files panel; the built-in sample if there are none. Returns pages."""
    pages = []
    for path in sorted(glob.glob("*.pdf")):
        for n, page in enumerate(PdfReader(path).pages, start=1):
            text = " ".join((page.extract_text() or "").split())
            if text:
                pages.append({"source": path, "page": n, "text": text})
    if not pages:
        for part in SAMPLE.split("=== page ")[1:]:
            n, text = part.split(" ===", 1)
            pages.append({"source": SAMPLE_NAME, "page": int(n), "text": " ".join(text.split())})
    return pages


def make_chunks(pages, words_per_chunk, overlap):
    """Cut each page into overlapping runs of words. Each chunk remembers where it came from."""
    chunks = []
    step = max(1, words_per_chunk - overlap)
    for p in pages:
        words = p["text"].split()
        for start in range(0, max(1, len(words) - overlap), step):
            piece = " ".join(words[start:start + words_per_chunk])
            if piece:
                chunks.append({"source": p["source"], "page": p["page"], "text": piece})
    return chunks


def search(question, chunks, vectors, k):
    q = embedder.encode([question], normalize_embeddings=True)[0]
    scores = vectors @ q
    best = np.argsort(scores)[::-1][:k]
    return [dict(chunks[i], score=float(scores[i])) for i in best]


def show_hits(hits):
    for n, h in enumerate(hits, start=1):
        print(f"[{n}]  score {h['score']:.2f}   {h['source']}, page {h['page']}")
        print(textwrap.fill(h["text"], 100, initial_indent="     ", subsequent_indent="     "))
        print()


def build_prompt(question, hits, strict=True, audience="", extra_rule=""):
    passages = "\n\n".join(f"[{n}] ({h['source']}, page {h['page']})\n{h['text']}"
                             for n, h in enumerate(hits, start=1))
    rules = ("Answer using ONLY the numbered passages below. If they do not contain the answer, "
             "reply exactly: The documents don't say.\nAfter each fact, cite the passage number "
             "you used, like [2].") if strict else "Use the passages below to help you answer."
    who = f"You are an assistant for {audience}, answering a question about some documents." if audience \
        else "You are answering a question about some documents."
    extra = f"\n{extra_rule}" if extra_rule else ""
    return f"{who}\n{rules}{extra}\n\nPASSAGES\n\n{passages}\n\nQUESTION: {question}"


def show_prompt(prompt):
    """A box to copy from. Click in it, select all, copy, paste into your AI."""
    if IN_COLAB:
        display(HTML(
            "<p><b>Click in the box → select all (Ctrl-A / ⌘-A) → copy (Ctrl-C / ⌘-C) → paste into your AI.</b></p>"
            f"<textarea readonly rows='16' style='width:100%;font-family:monospace;font-size:12px' "
            f"onclick='this.select()'>{html.escape(prompt)}</textarea>"))
    else:
        print(prompt)
    print(f"\n({len(prompt.split())} words, roughly {len(prompt) // 4} tokens)")


# Colab's built-in AI: no API key. A preview feature, so it may be missing or rate-limited on some
# accounts -- then ask_ai() returns None and you copy and paste the prompt instead.
try:
    from google.colab import ai as colab_ai
except Exception:
    colab_ai = None


def ask_ai(prompt):
    """Colab's built-in AI's answer, or None if it isn't available here."""
    if colab_ai is None:
        return None
    try:
        return colab_ai.generate_text(prompt).strip()
    except Exception as err:
        print(f"(Colab's built-in AI didn't answer: {err})")
        return None


def answer_or_copy(prompt):
    """Ask Colab's AI; if it can't, show the prompt to copy into your own AI."""
    reply = ask_ai(prompt)
    if reply:
        print("COLAB'S AI ANSWERED:\n")
        print(textwrap.fill(reply, 100))
    else:
        print("Colab's built-in AI isn't available — copy this prompt into your own AI instead.\n")
        show_prompt(prompt)


def check(ok, good, bad):
    print(("✅ " + good) if ok else ("❌ " + bad))


print("✅ Setup done — the embedding model is loaded.")
print("✅ Colab's built-in AI is available." if colab_ai else
      "ℹ️  Colab's built-in AI isn't available here — you'll copy and paste prompts into your own AI.")

## 5.1 · Load

In [ ]:
# ▶ DEMO 5.1.1 — read the documents
pages = load_documents()
sources = sorted({p["source"] for p in pages})
print(f"{len(sources)} document(s), {len(pages)} page(s), {sum(len(p['text'].split()) for p in pages):,} words")
for s in sources:
    print("  •", s)
print("\nThe start of the first page:\n")
print(textwrap.fill(pages[0]["text"][:600] + " …", 100))

If your PDF shows **0 words**, it is probably a scanned image with no real text in it. Try a
different version of the document, or use the sample.

## 5.2 · Chunk

An AI can only read so much at once, and we only want to hand it the **relevant** parts. So we cut
the documents into **chunks** — short runs of words — and search those.

In [ ]:
# 🔍 READ 5.2.1 — predict: what happens to the NUMBER of chunks if you make them bigger?
# And what might go wrong if chunks are tiny — say, 20 words?
words_per_chunk = 150  # @param {type:"slider", min:20, max:400, step:10}
overlap = 30           # @param {type:"slider", min:0, max:100, step:10}

chunks = make_chunks(pages, words_per_chunk, overlap)
print(f"{len(chunks)} chunks of up to {words_per_chunk} words (each overlapping the last by {overlap})\n")
for c in chunks[:3]:
    print(f"— {c['source']}, page {c['page']} —")
    print(textwrap.fill(c["text"], 100), "\n")

**Overlap** stops a fact being cut in half at a chunk boundary: the end of one chunk is repeated
at the start of the next.

## 5.3 · Embed

Every chunk becomes 384 numbers that capture its meaning — the same embedding model as Part 4.

In [ ]:
# ▶ DEMO 5.3.1 — turn every chunk into numbers (re-run this if you change the chunk size)
vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True)
print(f"{vectors.shape[0]} chunks × {vectors.shape[1]} numbers each")
print("The first chunk starts:", np.round(vectors[0][:6], 3), "…")

## 5.4 · Retrieve

Turn the question into numbers the same way, and find the chunks whose numbers are closest.

In [ ]:
# ✍️ YOU TRY 5.4.1 — ask a question about your documents. Look at what comes back.
question = "How many requirement changes never reached the test procedures?"  # @param {type:"string"}
top_k = 3  # @param {type:"slider", min:1, max:8, step:1}

hits = search(question, chunks, vectors, top_k)
show_hits(hits)

**Look before you go on:** is the answer actually in one of those passages? If not, nothing later
can fix it. Try rewording the question, or a bigger `top_k`.

## 5.5 · Build the prompt

This is the whole trick of RAG: it's **just a prompt**. The passages, some rules, and the question.

In [ ]:
# ▶ DEMO 5.5.1 — the prompt the AI will get. Read it: it's the passages, some rules, and your question.
prompt = build_prompt(question, hits)
show_prompt(prompt)

## 5.6 · Get the answer

**5.6.1** sends the prompt to Colab's built-in AI. If that isn't available on your account, copy the
prompt from 5.5 into your own AI and paste its answer into **5.6.2**.

In [ ]:
# ▶ DEMO 5.6.1 — send the prompt to Colab's built-in AI
auto_answer = ask_ai(prompt)
if auto_answer:
    print(textwrap.fill(auto_answer, 100))
else:
    print("Colab's built-in AI isn't available here. Copy the prompt from 5.5 into your own AI,")
    print("then paste its answer into 5.6.2.")

In [ ]:
# ✅ CHECKPOINT 5.6.2 — only if 5.6.1 didn't answer: paste your AI's answer between the triple quotes.
pasted = """
"""

answer = pasted.strip() or auto_answer or ""
ai_used = "pasted" if pasted.strip() else ("Colab's AI" if auto_answer else "none yet")
print("QUESTION:", question)
print(f"ANSWER ({ai_used}):", answer or "(nothing yet)")
check(len(answer) > 0 and "[" in answer,
      "Answered, with citations. Check them: does the passage it cites really say that?",
      "No answer yet — or no [1]-style citations, which means the AI ignored the rules. Note that too.")

**You built a RAG.** Load → chunk → embed → retrieve → prompt → answer. Every "chat with your
documents" product is this pipeline, with more polish.

**Next — D01.06, Break It, Then Ship It (Part 6):** ten test questions, and which step fails.

# Part 6 · Break It, Then Ship It
**Deck D01.06** · 3:15

One good answer proves nothing. Test the RAG you built in Part 5, find which step fails, then
turn it into your team's prototype.

In [ ]:
# @title ⚙️ PART 6 SETUP — run this first, and again if Colab reconnects { display-mode: "form" }
# Part 6 uses the RAG from Part 5. If Colab reconnected over the break, this rebuilds it.
if "load_documents" not in globals():
    print("❌ Scroll up and run the PART 5 SETUP cell first, then run this one again.")
else:
    if "vectors" not in globals():
        words_per_chunk, overlap, top_k = 150, 30, 3
        pages = load_documents()
        chunks = make_chunks(pages, words_per_chunk, overlap)
        vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True)
        print("Rebuilt the RAG with the default settings (150-word chunks, top_k 3).")
    print(f"✅ Ready — {len(chunks)} chunks from {len({c['source'] for c in chunks})} document(s), top_k = {top_k}.")

## 6.1 · A test set

One good answer proves nothing. Write **ten questions** and score every one:

- **5 the documents answer** — the right answer is in there
- **3 they don't** — the right answer is *"The documents don't say"*
- **2 tricky ones** — vague, needs two passages, or where the document contradicts itself

The ten below are for the sample report. **Replace them with your own** if you're using your team's
documents.

In [ ]:
# ✍️ YOU TRY 6.1.1 — your ten questions
tests = [
    # the documents answer these
    "How many requirements changed after the critical design review?",
    "How many units were in the lifetime buy of the end-of-life chip?",
    "How long did it take to reverse-engineer the legacy interface?",
    "How much did the rework cost?",
    "Why did the AI assistant give wrong answers in daily use?",
    # the documents don't answer these
    "What is the program's total contract value?",
    "Who was the program manager?",
    "When does Phase C start?",
    # tricky
    "How many months did the schedule slip?",
    "Should the AI assistant be used operationally?",
]
print(len(tests), "questions")

In [ ]:
# ▶ DEMO 6.1.2 — ask all ten at once with Colab's built-in AI (about a minute)
answers = []
for n, q in enumerate(tests, start=1):
    hits = search(q, chunks, vectors, top_k)
    reply = ask_ai(build_prompt(q, hits))
    if reply is None:
        print("Colab's built-in AI isn't available here — use 6.1.3 to do them one at a time.")
        answers = []
        break
    answers.append(reply)
    pages_used = ", ".join(str(h["page"]) for h in hits)
    print(f"Q{n}: {q}\n    passages from page(s) {pages_used}")
    print(textwrap.fill(reply, 96, initial_indent="    → ", subsequent_indent="      "), "\n")

In [ ]:
# ✍️ YOU TRY 6.1.3 — one at a time (to see the passages, or if 6.1.2 couldn't ask the AI).
# Pick a number, run, and if you need to, copy the prompt into your own AI.
question_number = 1  # @param {type:"slider", min:1, max:10, step:1}

q = tests[question_number - 1]
print(f"Q{question_number}: {q}\n")
hits = search(q, chunks, vectors, top_k)
show_hits(hits)
answer_or_copy(build_prompt(q, hits))

## 6.2 · Score it

For each question, read the AI's answer **and the passages it was given** (6.1.3 shows them), then pick one verdict:

| Verdict | Means |
| --- | --- |
| `right` | correct, and the citation supports it |
| `said don't know` | it said *"The documents don't say"* — **right** if the answer really isn't there |
| `retrieval missed` | the answer is in the documents, but not in the passages it was given |
| `made it up` | the passages were fine; the AI said something they don't say |
| `doc was wrong` | it faithfully repeated something the document itself gets wrong or contradicts |

No failures? Make some. Set `top_k` to **1**, or chunks to **20 words**, and re-ask. Then set
`top_k` to **5** and re-ask the schedule question: does a second, different number turn up?

In [ ]:
# ✍️ YOU TRY 6.2.1 — one verdict per question, in order (copy the words from the table)
verdicts = [
    "",  # Q1
    "",  # Q2
    "",  # Q3
    "",  # Q4
    "",  # Q5
    "",  # Q6
    "",  # Q7
    "",  # Q8
    "",  # Q9
    "",  # Q10
]

In [ ]:
# ▶ DEMO 6.2.2 — the scoreboard
KINDS = ["right", "said don't know", "retrieval missed", "made it up", "doc was wrong"]
scored = [v for v in verdicts if v]
unknown = [v for v in scored if v not in KINDS]
if unknown:
    print("❌ Not a verdict from the table:", unknown)

counts = [scored.count(k) for k in KINDS]
good = scored.count("right") + scored.count("said don't know")
print(f"{len(scored)} of {len(tests)} scored.  Good answers: {good} of {len(scored)}")
print("(A \"said don't know\" only counts as good if the answer really isn't in the documents.)")

plt.figure(figsize=(9, 3.5))
plt.barh(KINDS[::-1], counts[::-1], color=["crimson", "darkorange", "goldenrod", "seagreen", "seagreen"])
plt.xlabel("questions"); plt.title("where did it go wrong?"); plt.show()

**Talk at your table:**
1. Which failure was most common? Is it the **search** (retrieval missed) or the **AI** (made it up)?
2. What one change would fix the most failures — smaller chunks? a bigger `top_k`? better documents?
3. Would you trust this to answer **your customers**? What would have to be true first?

## ⭐ 6.3 · Build (optional)

Pick one and re-score the questions it affects:

- **Change the chunk size** in 5.2 (then re-run 5.3) — does *retrieval missed* go down?
- **Same prompt, two AIs** — compare Colab's answer with one you paste into a different assistant. Do they agree?
- **Loosen the rules** — run the cell below, which drops "use ONLY the passages". Ask a
  question the documents *don't* answer. What happens now?

In [ ]:
# ✍️ YOU TRY 6.3.1 — the same question with the rules loosened
loose_question = "What is the program's total contract value?"  # @param {type:"string"}
answer_or_copy(build_prompt(loose_question, search(loose_question, chunks, vectors, top_k), strict=False))

## 6.4 · Make it yours (the sprint)

Your prototype is this RAG, aimed at **one kind of person** with **one job to do**. Say who it's
for and add one rule that suits them, then use your three best questions from the test set —
including **one it gets wrong** — for the demo.

In [ ]:
# ✍️ YOU TRY 6.4.1 — who is it for, and what should it always do for them?
who_its_for = "a program manager starting Phase C"  # @param {type:"string"}
extra_rule = "Answer in two sentences of plain language, then name the recommendation that applies."  # @param {type:"string"}
demo_question = "Which lesson would have saved the most schedule?"  # @param {type:"string"}

answer_or_copy(build_prompt(demo_question, search(demo_question, chunks, vectors, top_k),
                            audience=who_its_for, extra_rule=extra_rule))

## Part 6 wrap

You can now:
- name the steps of a RAG pipeline — **load, chunk, embed, retrieve, prompt, answer** — and say what each produces
- explain why RAG answers can still be wrong, and **which step** failed
- test an AI system with a **test set** instead of trusting one good answer

This RAG is your team's starting prototype for the sprint.